## Task 1 — Simulating Sales Data (Slide 17)

I generate random integer sales (10–100 units) for 364 days × 4 products. `randint`'s upper bound is exclusive, so `high=101` is needed to include 100. The seed makes the results reproducible.

In [52]:
import numpy as np

np.random.seed(42)
num_days = 364
num_products = 4
daily_sales = np.random.randint(low=10, high=101, size=(num_days, num_products))

print("Shape:", daily_sales.shape)
print("First 5 days:\n", daily_sales[:5])

Shape: (364, 4)
First 5 days:
 [[61 24 81 70]
 [30 92 96 84]
 [84 97 33 12]
 [31 62 11 97]
 [39 47 11 73]]


**Result:** the array has shape `(364, 4)` — one row per day, one column per product.

## Task 2 — Computing Monthly Averages (Slide 18)

For this task we take a separate dataset of 360 days (12 months × 30 days) for 4 products. I reshaped it to (12, 30, 4), standing for months, days per month, and products, and took the mean over axis=1, the days axis.

In [54]:
num_days_360 = 360
num_products_360 = 4
daily_sales_360 = np.random.randint(low=10, high=101, size=(num_days_360, num_products_360))
monthly_sales = daily_sales_360.reshape(12, 30, 4)
monthly_avg_sales = monthly_sales.mean(axis=1)

print("Shape:", monthly_avg_sales.shape)
print("Average daily sales per product, first 3 months:\n", monthly_avg_sales[:3].round(2))

Shape: (12, 4)
Average daily sales per product, first 3 months:
 [[56.57 51.9  49.97 56.2 ]
 [58.37 49.97 60.07 53.8 ]
 [48.93 55.67 60.27 53.93]]


**Result:** shape `(12, 4)`. Averaging over axis 1 collapses the 30 days, leaving months and products. Most values are near 55, which is what we expect since the mean of uniform values from 10 to 100 is 55 (individual months vary around it).

## Task 3 — Performance Comparison (Slide 19)

We compute the total sales over all days and products in two ways: nested Python loops and NumPy's vectorized `np.sum`. First I check that both give the same answer, then time them.

In [53]:
def sum_with_loop(data):
    total = 0
    for row in data:
        for item in row:
            total += item
    return total

print("Loop total:      ", sum_with_loop(daily_sales))
print("np.sum total:    ", np.sum(daily_sales))
print("Same result:     ", sum_with_loop(daily_sales) == np.sum(daily_sales))

Loop total:       78055
np.sum total:     78055
Same result:      True


In [55]:
%timeit sum_with_loop(daily_sales)

318 μs ± 26.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [57]:
%timeit np.sum(daily_sales)

4.03 μs ± 705 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


**Result:** both approaches give the same total, but `np.sum` is much faster (my measured times: loop ≈ 318 μs , `np.sum` ≈ 4 μs; speedup ≈ 80x). The loop is slow because Python handles one element at a time with interpreter overhead, while NumPy runs the whole sum in optimized compiled C code over contiguous memory.

## Task 4 — Weekly Sales Aggregation (Slide 20)

364 days = 52 weeks × 7 days, so the full array reshapes cleanly to `(52, 7, 4)` (weeks, days per week, products). Summing over axis 1 gives weekly totals per product.

In [ ]:
weekly_sales_data = daily_sales.reshape(52, 7, 4)
total_weekly_sales = weekly_sales_data.sum(axis=1)

print("Shape of total_weekly_sales:", total_weekly_sales.shape)
print("Total sales for the first week:")
print(total_weekly_sales[0])

Shape of total_weekly_sales: (52, 4)
Total sales for the first week:
[381 383 372 479]


In [14]:
print(np.array_equal(total_weekly_sales[0], daily_sales[:7].sum(axis=0)))

True


**Result:** shape `(52, 4)` — total units per product for each of the 52 weeks. The sanity check returns `True`, confirming the reshape grouped days into weeks correctly.